In [33]:
from llama_index.core import SimpleDirectoryReader

input_dirs = {
    "roo-coder": "../TaskClient/docs/roo-docs", 
    "cline": "../TaskClient/docs/cline-logs"
}
documents = []

for agent_name, path in input_dirs.items():
    # פונקציית עזר שמוסיפה מטא-דאטה לכל קובץ שנקרא מהתיקייה
    def add_agent_metadata(file_path):
        return {
            "agent_tool": agent_name,
            "source_folder": path
        }

    reader = SimpleDirectoryReader(
        input_dir=path, 
        recursive=True, 
        required_exts=[".md"],
        file_metadata=add_agent_metadata # כאן קורה הקסם
    )
    documents.extend(reader.load_data())

print(f"נטענו {len(documents)} מסמכים עם תיוג Agent.")

נטענו 9 מסמכים עם תיוג Agent.


In [1]:
from llama_index.core import SimpleDirectoryReader
input_dirs = ["../TaskClient/docs/roo-docs", "../TaskClient/docs/cline-logs"]
documents = []

for d in input_dirs:
    # כאן d הוא מחרוזת בודדת בכל פעם, אז זה יעבוד
    reader = SimpleDirectoryReader(
        input_dir=d, 
        recursive=True, 
        required_exts=[".md"]
    )
    documents.extend(reader.load_data())

In [34]:
from llama_index.core.node_parser import MarkdownNodeParser

# יצירת הפארסר שמזהה כותרות (H1, H2, וכו')
parser = MarkdownNodeParser()

# פירוק המסמכים לצמתים מבוססי מבנה
nodes = parser.get_nodes_from_documents(documents)

# טיפ: כדאי לבדוק את ה-metadata של הצמתים שנוצרו
# כל צומת יכיל כעת מידע על ה-Header שהוא שייך אליו
print(nodes[0].metadata)

{'agent_tool': 'roo-coder', 'source_folder': '../TaskClient/docs/roo-docs', 'header_path': '/'}


In [35]:

from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.core.node_parser import MarkdownNodeParser
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("COHERE_API_KEY")
# 1. הגדרת מודל ה-Embedding של Cohere
# המודל 'embed-multilingual-v3.0' מעולה למסמכים טכניים בעברית ובאנגלית
embed_model = CohereEmbedding(
    cohere_api_key=api_key,
    model_name="embed-multilingual-v3.0",
)

# 2. עדכון הגדרות הגלובליות של LlamaIndex
Settings.embed_model = embed_model
# כאן כדאי להגדיר גם את ה-chunk_size אם את לא משתמשת ב-Parser חיצוני
Settings.chunk_size = 512 

# 3. פירוק לצמתים (Nodes) באמצעות MarkdownNodeParser כפי שסיכמנו
parser = MarkdownNodeParser()
nodes = parser.get_nodes_from_documents(documents)

# 4. יצירת האינדקס הוקטורי
# בשלב זה המערכת שולחת את הטקסט ל-Cohere ומקבלת וקטורים בחזרה
index = VectorStoreIndex(nodes)

print(f"הסתיים אינדוקס של {len(nodes)} צמתים.")

2026-03-18 17:57:17,908 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 17:57:18,910 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 17:57:19,805 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 17:57:20,931 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 17:57:22,206 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 17:57:23,234 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 17:57:24,038 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 17:57:25,028 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 17:57:25,977 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-03-18 17:57:27,187 - INFO - HTTP Request: POST https://api.cohere.com/v2/embe

הסתיים אינדוקס של 119 צמתים.


In [42]:
import os
from pinecone import Pinecone
from llama_index.vector_stores.pinecone import PineconeVectorStore

# 1. הגדר את המפתח כאן (החלף את המחרוזת pcsk_... במפתח האמיתי שלך)
MY_API_KEY = os.getenv("PINECONE_API_KEY")

# 2. התחברות ל-Pinecone
pc = Pinecone(
    api_key=MY_API_KEY, 
    ssl_verify=False
)

# 3. הגדרת ה-Index Host והאינדקס
INDEX_HOST = "https://rag-index-2mwyo1g.svc.aped-4627-b74a.pinecone.io"
pinecone_index = pc.Index(name="rag-index", host=INDEX_HOST)

# 4. הגדרת ה-Vector Store - שים לב להעברת המפתח גם כאן!
vector_store = PineconeVectorStore(
    pinecone_index=pinecone_index,
    api_key=MY_API_KEY  # חשוב להעביר את המפתח גם ל-LlamaIndex
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 5. יצירת האינדקס
index = VectorStoreIndex(
    nodes, 
    storage_context=storage_context,
    show_progress=True
)

Generating embeddings:   0%|          | 0/119 [00:00<?, ?it/s]

2026-03-18 18:55:45,111 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
Upserted vectors: 100%|██████████| 119/119 [00:21<00:00,  5.61it/s]


In [37]:
from llama_index.core import get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.llms.cohere import Cohere
from llama_index.core.postprocessor import SimilarityPostprocessor
# 1. הגדרת ה-LLM (לניסוח התשובה הסופית)
from llama_index.core import Settings

from llama_index.llms.cohere import Cohere
from llama_index.core import Settings

# האופציה המומלצת - מודל command-r בגרסה מעודכנת
llm = Cohere(
    api_key=os.getenv("COHERE_API_KEY"), 
    model="command-r-08-2024" # או פשוט "command-r-v2" בהתאם לעדכוני Cohere האחרונים
)

# הגדרה גלובלית
Settings.llm = llm

# חשוב: אם כבר יצרת את ה-query_engine, צריך ליצור אותו מחדש 
# כדי שהוא "יידע" להשתמש ב-llm החדש:
query_engine = index.as_query_engine(
    similarity_top_k=5,
    llm=llm # הזרקה ישירה של המודל המעודכן
)

# 2. הגדרת ה-Retriever: כמה צמתים לשלוף מ-Pinecone?
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=5, # שולף את 5 ה-chunks הכי רלוונטיים
)

# 3. הגדרת ה-Synthesizer: איך "להלחים" את המידע לתשובה
response_synthesizer = get_response_synthesizer(
    llm=llm,
    response_mode="compact" # משלב את ה-chunks בצורה חסכונית ב-tokens
)


# 1. נוריד את הסף ל-0.45 כדי לאפשר לצמתים שמצאנו לעבור
# או שפשוט נסיר את ה-Postprocessor זמנית לבדיקה
node_postprocessors = [
    SimilarityPostprocessor(similarity_cutoff=0.45) 
]

# 2. עדכון ה-Query Engine עם הסף החדש
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=node_postprocessors
)

In [ ]:
import gradio as gr

def ask_rag(question, history):
    try:
        # history מכיל את היסטוריית הצ'אט, כרגע אנחנו מתמקדים רק בשאלה הנוכחית
        response = query_engine.query(question)
        return str(response)
    except Exception as e:
        return f"אופס, קרתה שגיאה: {str(e)}"

# יצירת ממשק הצ'אט ללא פרמטר theme (או שימוש בערך ברירת המחדל)
demo = gr.ChatInterface(
    fn=ask_rag,
    title="🤖 Agentic Doc Explorer",
    description="שאל שאלות על תהליכי העבודה של כלי ה-AI (Cursor, Roo, Cline)",
    examples=["מה הצבע העיקרי שנבחר לדיזיין?", "אילו שדות נוספו ל-DB?"]
)

if __name__ == "__main__":
    # אם אתה עובד ב-Jupyter/Colab, מומלץ להוסיף inline=False אם הלשונית לא נפתחת
    demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7870


2026-03-18 18:08:07,814 - INFO - HTTP Request: GET http://127.0.0.1:7870/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-18 18:08:07,847 - INFO - HTTP Request: HEAD http://127.0.0.1:7870/ "HTTP/1.1 200 OK"
2026-03-18 18:08:07,902 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-03-18 18:08:09,237 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-18 18:08:09,318 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
2026-03-18 18:08:10,650 - INFO - HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe "HTTP/1.1 200 OK"
2026-03-18 18:10:18,743 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026-03-18 18:10:21,008 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-error-analytics "HTTP/1.1 200 OK"
2026-03-18 18:10:21,042 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-03-18 18:10:26,809 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


In [ ]:
# נסי שאלה שאת יודעת שקיימת בתיעוד שלך
test_query = "מה הצבע העיקרי?" 

# שליפה ידנית של ה-Nodes
retrieved_nodes = retriever.retrieve(test_query)

print(f"נמצאו {len(retrieved_nodes)} צמתים רלוונטיים.")
for i, node in enumerate(retrieved_nodes):
    print(f"--- Node {i+1} (Score: {node.score:.2f}) ---")
    print(node.text[:200] + "...") # הדפסת תחילת הטקסט

In [ ]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

def ask_rag_with_filter(question, agent_name, history):
    # יצירת סינון אם נבחר כלי ספציפי
    filters = None
    if agent_name and agent_name != "הכל":
        filters = MetadataFilters(
            filters=[ExactMatchFilter(key="agent_tool", value=agent_name)]
        )
    
    # הרצת השאילתה עם הפילטר
    response = query_engine.query(question, filters=filters)
    return str(response)

# עדכון ממשק Gradio עם Dropdown
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# 🤖 Agentic Doc Explorer")
    with gr.Row():
        agent_selector = gr.Dropdown(["הכל", "roo-coder", "cline"], label="בחר כלי (Agent)", value="הכל")
    
    chatbot = gr.ChatInterface(
        fn=ask_rag_with_filter,
        additional_inputs=[agent_selector]
    )

demo.launch()

In [38]:
from llama_index.core.workflow import Event
from llama_index.core.schema import NodeWithScore
from typing import List

class RetrievalEvent(Event):
    """אירוע שנשלח לאחר ולידציה של השאילתה"""
    query: str

class ValidationEvent(Event):
    """אירוע שמעביר את תוצאות החיפוש לבדיקת איכות"""
    nodes: List[NodeWithScore]
    query: str

class GenerationEvent(Event):
    """אירוע שמאשר שהמידע איכותי ומוכן לניסוח תשובה"""
    nodes: List[NodeWithScore]
    query: str

In [39]:
from llama_index.core.workflow import Workflow, step, StartEvent, StopEvent

class AgenticRAGWorkflow(Workflow):
    def __init__(self, retriever, response_synthesizer, **kwargs):
        super().__init__(**kwargs)
        # הזרקת הכלים מה-MVP לתוך ה-Workflow
        self.retriever = retriever
        self.synthesizer = response_synthesizer

    @step
    async def validate_query(self, ev: StartEvent) -> RetrievalEvent | StopEvent:
        """בדיקה ראשונית: האם השאלה תקינה?"""
        query = ev.get("query")
        if not query or len(query.strip()) < 3:
            return StopEvent(result="אופס! השאילתה קצרה מדי. נסה לשאול שאלה מפורטת יותר.")
        return RetrievalEvent(query=query)

    @step
    async def run_retrieval(self, ev: RetrievalEvent) -> ValidationEvent:
        """ביצוע החיפוש הוקטורי (Retriever מה-MVP)"""
        nodes = self.retriever.retrieve(ev.query)
        return ValidationEvent(nodes=nodes, query=ev.query)

    @step
    async def validate_results(self, ev: ValidationEvent) -> GenerationEvent | StopEvent:
        """בדיקת איכות: האם מצאנו מידע אמין?"""
        # ולידציה על חוסר תוצאות או Confidence נמוך
        if not ev.nodes or all(n.score < 0.45 for n in ev.nodes):
            return StopEvent(result="לא מצאתי מידע מספיק אמין בתיעוד כדי לענות על כך בוודאות.")
        
        return GenerationEvent(nodes=ev.nodes, query=ev.query)

    @step
    async def generate_answer(self, ev: GenerationEvent) -> StopEvent:
        """ניסוח התשובה הסופית (Synthesizer מה-MVP)"""
        response = self.synthesizer.synthesize(query=ev.query, nodes=ev.nodes)
        return StopEvent(result=str(response))

In [40]:
# יצירת המופע עם האובייקטים מה-MVP שלך
# (וודאי שהרצת את קוד ה-MVP קודם לכן באותו Notebook)
rag_workflow = AgenticRAGWorkflow(
    retriever=retriever, 
    response_synthesizer=response_synthesizer, 
    timeout=60, 
    verbose=True
)

# פונקציה עבור Gradio
async def run_agent_chat(message, history):
    result = await rag_workflow.run(query=message)
    return str(result)